In [14]:
import re
import random
import os

import torch
import torch.nn as nn
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from collections import Counter

device = torch.device('mps' if torch.mps.is_available() else 'cpu')
print('device:', device)

device: mps


In [17]:
with open('./model_file.pkl', 'rb') as f:
    vocab = pickle.load(f)

In [18]:
def simple_tokenize(text : str) -> list:
    text = text.lower()
    text = re.sub(r"[^가-힣ㄱ-ㅎㅏ-ㅣa-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    tokens = text.split()
    return tokens


In [19]:
def encode_text(text, vocab, max_len=50):
    tokens = simple_tokenize(text)
    token_ids = [vocab.get(token, vocab['<UNK>']) for token in tokens]
    if len(token_ids) > max_len:
        token_ids = token_ids[:max_len]
    else:
        token_ids += [vocab['<PAD>']] * (max_len - len(token_ids))
    return token_ids

In [11]:
def predict_sentiment(text, model, vocab, device, max_len=50):
    model.eval()
    input_ids = encode_text(text, vocab, max_len=max_len)
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)


    with torch.no_grad():
        logits = model(input_tensor)
        prob = torch.sigmoid(logits).item()
    label = 1 if prob >= 0.5 else 0
    return label, prob


In [20]:
class SentimentBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_layers=1, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,       # [배치, 길이, 특징]
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)
    
    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        output, (hidden, cell) = self.lstm(embedded)

        forward_hidden = hidden[-2]     # 마지막 층의 정방향 은닉 상태
        backward_hidden = hidden[-1]    # 마지막 층의 역방향 은닉 상태
        final_hidden = torch.cat((forward_hidden, backward_hidden), dim=1)
        final_hidden = self.dropout(final_hidden)
        logits = self.fc(final_hidden).squeeze(1)

        return logits

In [21]:
model = SentimentBiLSTM(
    vocab_size=len(vocab),
    embed_dim=128,
    hidden_dim=128,
    num_layers=1,
    dropout=0.3
).to(device)

In [23]:
model.load_state_dict(torch.load('./best_sentiment_bilstm.pt', map_location=device))

<All keys matched successfully>

In [34]:
predict_sentiment('', model=model, vocab=vocab, device=device, max_len=50)

(1, 0.5178844928741455)

In [10]:
import plotly.graph_objects as go
import pandas as pd
df = pd.read_csv("./005930.csv")

df.날짜 = pd.to_datetime(df.날짜, format="%Y%m%d")


In [11]:
fig = go.Figure()

In [12]:
fig.add_trace(go.Candlestick(
    x= df.날짜,
    open=df.시가 ,
    high=df.고가 ,
    low=df.저가,
    close=df.종가
))


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'close': {'bdata': ('TPICAJT/AgBENgMA4BwDALAkAwAoEQ' ... 'QAsBgFAITMBAAwXQQA/LgEADDaBAA='),
                        'dtype': 'i4'},
              'high': {'bdata': ('KPgCAAQXAwDkRQMAjCoDADg4AwD4GA' ... 'UA2DsFADgsBQCAowQAqMYEAIj1BAA='),
                       'dtype': 'i4'},
              'low': {'bdata': ('lOYCAJDvAgCMKgMAEBUDALwiAwA4Bg' ... 'QA6OUEAMzABACcSwQAbFMEAJifBAA='),
                      'dtype': 'i4'},
              'open': {'bdata': ('tOkCABAVAwDwQwMAmCgDAHQuAwA4Bg' ... 'UArO8EAKQaBQDQbAQA9GYEAADiBAA='),
                       'dtype': 'i4'},
              'type': 'candlestick',
              'x': array(['2026-04-06T00:00:00.000000', '2026-04-07T00:00:00.000000',
                          '2026-04-08T00:00:00.000000', '2026-04-09T00:00:00.000000',
                          '2026-04-10T00:00:00.000000', '2026-04-13T00:00:00.000000',
                          '2026-04-14T00:00:00.000000', '2026-04-15T00:00:00.000000',
                          '2026-04-16T00:00:00.000000', '2026-04-17T00:00:00.000000',
                          '2026-04-20T00:00:00.000000', '2026-04-21T00:00:00.000000',
                          '2026-04-22T00:00:00.000000', '2026-04-23T00:00:00.000000',
                          '2026-04-24T00:00:00.000000', '2026-04-27T00:00:00.000000',
                          '2026-04-28T00:00:00.000000', '2026-04-29T00:00:00.000000',
                          '2026-04-30T00:00:00.000000', '2026-05-04T00:00:00.000000',
                          '2026-05-06T00:00:00.000000', '2026-05-07T00:00:00.000000',
                          '2026-05-08T00:00:00.000000', '2026-05-11T00:00:00.000000',
                          '2026-05-12T00:00:00.000000', '2026-05-13T00:00:00.000000',
                          '2026-05-14T00:00:00.000000', '2026-05-15T00:00:00.000000',
                          '2026-05-18T00:00:00.000000', '2026-05-19T00:00:00.000000',
                          '2026-05-20T00:00:00.000000', '2026-05-21T00:00:00.000000',
                          '2026-05-22T00:00:00.000000', '2026-05-26T00:00:00.000000',
                          '2026-05-27T00:00:00.000000', '2026-05-28T00:00:00.000000',
                          '2026-05-29T00:00:00.000000', '2026-06-01T00:00:00.000000',
                          '2026-06-02T00:00:00.000000', '2026-06-04T00:00:00.000000',
                          '2026-06-05T00:00:00.000000', '2026-06-08T00:00:00.000000',
                          '2026-06-09T00:00:00.000000', '2026-06-10T00:00:00.000000',
                          '2026-06-11T00:00:00.000000', '2026-06-12T00:00:00.000000',
                          '2026-06-15T00:00:00.000000', '2026-06-16T00:00:00.000000',
                          '2026-06-17T00:00:00.000000', '2026-06-18T00:00:00.000000',
                          '2026-06-19T00:00:00.000000', '2026-06-22T00:00:00.000000',
                          '2026-06-23T00:00:00.000000', '2026-06-24T00:00:00.000000',
                          '2026-06-25T00:00:00.000000', '2026-06-26T00:00:00.000000',
                          '2026-06-29T00:00:00.000000', '2026-06-30T00:00:00.000000',
                          '2026-07-01T00:00:00.000000', '2026-07-02T00:00:00.000000',
                          '2026-07-03T00:00:00.000000', '2026-07-06T00:00:00.000000'],
                         dtype='datetime64[us]')}],
    'layout': {'template': '...'}
})

In [1]:
from ultralytics import YOLO
import cv2
from ultralytics.utils.plotting import Annotator
model = YOLO("yolov8m-pose.pt")






def predict(frame, iou=0.7, conf=0.25):
    results = model(source=frame,
            device='cpu',
            iou=iou ,
            conf=conf ,
            verbose=False,
            )
    return results[0]




def draw_boxes(result, frame):
    for boxes in result.boxes:
        x1, y1, x2, y2, score, classes = boxes.data.squeeze().cpu().numpy()
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), 1)
    return frame


def draw_keypoints(result, frame):
    annotator = Annotator(frame, line_width=1)
    for kps in result.keypoints:
        kps = kps.data.squeeze()
        annotator.kpts(kps)
       
        nkps = kps.cpu().numpy()
        # nkps[:,2] = 1
        # annotator.kpts(nkps)
        for idx, (x, y, score) in enumerate(nkps):
            if score > 0.5:
                cv2.circle(frame, (int(x), int(y)), 3, (0, 0, 255), cv2.FILLED)
                cv2.putText(frame, str(idx), (int(x), int(y)), cv2.FONT_HERSHEY_COMPLEX, 1, (0, 0, 255), 1)
       
    return frame


capture = cv2.VideoCapture(0)
capture.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
capture.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
while True:
    ret, frame = capture.read()
    frame = cv2.flip(frame, 1)
    result = predict(frame)
    frame = draw_boxes(result, frame)
    frame = draw_keypoints(result, frame)
    frame = cv2.flip(frame, 1)




    # cv2.putText(frame, text, position, font, scale, color, thickness)
    if not ret:
        print("카메라 오류")
        break
    # print(type(frame))
    cv2.imshow("VideoFrame", frame)




    if cv2.waitKey(10) & 0xFF == ord('q'):
        capture.release()
        cv2.destroyAllWindows()
        break


WARNING ⚠️ Download failure, retrying 1/3 https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8m-pose.pt... <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)>


######################################################################## 100.0%


KeyboardInterrupt: 

In [6]:
import cv2
from ultralytics import YOLO
import numpy as np

img = cv2.imread('./image.jpg')
model = YOLO("yolov8l-pose.pt")

def get_class_colors(names):
    np.random.seed(42)
    return {i: tuple(int(c) for c in np.random.randint(50, 255, 3)) for i in names}




def draw_detections(annotated, boxes, names, colors):
    for box in boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        label = f"{names[cls_id]} {conf:.2f}"
        color = colors[cls_id]


        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
        cv2.rectangle(annotated, (x1, y1 - th - 8), (x1 + tw + 4, y1), color, -1)
        cv2.putText(annotated, label, (x1 + 2, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
img.shape
results = model(img, verbose=False, conf=0.5)
r = results[0]
names = model.names
colors = get_class_colors(names)
annotated = img.copy()
if r.boxes is not None and len(r.boxes) > 0:
    draw_detections(annotated, r.boxes, names, colors)
if r.boxes is not None and len(r.boxes) > 0:
    print(f"\n탐지된 객체: {len(r.boxes)}개")
    print("-" * 50)
    counts = {}
    for box in r.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        name = names[cls_id]
        counts[name] = counts.get(name, 0) + 1
        print(f"  {name:20s} conf={conf:.2f}  bbox=({x1},{y1})-({x2},{y2})")
    print("-" * 50)
    print("클래스별 집계:")
    for name, count in sorted(counts.items(), key=lambda x: -x[1]):
        print(f"  {name:20s} {count}개")
else:
    print("탐지된 객체가 없습니다.")


result = model(img, verbose=False)
r = result[0]


탐지된 객체: 5개
--------------------------------------------------
  person               conf=0.95  bbox=(2414,796)-(3238,2230)
  person               conf=0.95  bbox=(3125,714)-(4000,2231)
  person               conf=0.94  bbox=(207,664)-(1087,2225)
  person               conf=0.91  bbox=(1105,894)-(1783,2233)
  person               conf=0.90  bbox=(1683,852)-(2536,2227)
--------------------------------------------------
클래스별 집계:
  person               5개


In [2]:
from fastapi import FastAPI
from pydantic import BaseModel

In [4]:
app = FastAPI()

class PredictRequest(BaseModel):
    text: str

@app.get('/')
def root():
    return {'message': '처음 api 실행'}

@app.post('/predict')
def predict(request: PredictRequest):
    text = request.text

    return {'result': 'test'}